# 介绍
- <b>rollout是RL流水线里唯一能让“智能体” 与“环境”真正交互的环节</b>
- 后续所有的训练-》拟合-》梯度都只是对这段互动录像的统计

In [ ]:
import torch,gym
from transformers import AutoTokenizer,AutoModelForCauselLM

env = gym.make('CartPole-v1') # 可选择任何环境
tok = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')
model.eval() # rollout阶段不更新权重

def single_rollout(policy_net,max_steps=500):
    obs = env.reset()
    print(obs)
    traj = []
    for t in range(max_steps):
        # 1 状态 -> token id（简单映射示例）
        prompt = f"CartPole state {obs}"
        inputs = tok(prompt,return_tensor='pt')
        with torch.no_grad():
            logits = policy_net(**inputs).logits[:,-1,:] # 最后一个位置
        probs = logits.softmax(-1)
        # 按概率采样
        action = torch.multinomial(probs,1).item()
        logp = logits.log_softmax(-1)[0,action].item()
        
        # 2 与环境交互
        obs2,reward,done,_ = env.step(action)
        
        # 3 记录
        traj.append(dict(s=obs,a=action,r=reward,logp=logp,mask=1-done))
        obs = obs2
        
        if done:
            break
    return traj



# 并行rollout

- 大模型 RL 里常用 vLLM / DeepSpeed-FastGen 把 N×K 条 rollout 一次性推理出来，显存利用率 ↑10×。


In [ ]:
from concurrent.futures import ProcessPoolExecutor

def batch_rollout(policy_state_dict,N):
    # 把权重发给子进程，避免全局模型拷贝
    with ProcessPoolExecutor() as pool:
        trajs = pool.map(single_rollout,[policy_state_dict] * N)
    return list(trajs)